# Argus — `raw_videos_binary/`: a 2-class copy of the raw dataset

Standalone Colab utility. Run it top to bottom, once, with the Argus Drive mounted.

It copies `dataset/raw_videos/subject_*/level_<1-3>_clip_<N>.mp4` into a **new parallel tree**
`dataset/raw_videos_binary/subject_*/level_<1-2>_clip_<N>.mp4`, collapsing the 3 drowsiness
classes to 2:

| source level (`raw_videos/`) | this tree | class |
|---|---|---|
| 1  Alert         | **level_1** | Not Drowsy |
| 2  Low Vigilant  | **level_1** | Not Drowsy |
| 3  Drowsy        | **level_2** | Drowsy |

`Alert` and `Low Vigilant` both collapse into `level_1` (`Not Drowsy`); `Drowsy` becomes
`level_2`. This is the `drowsy_vs_not` binary framing. `raw_videos/` is never modified — just
delete `raw_videos_binary/` to undo. New recordings keep landing in `raw_videos/` as
`level_1/2/3`; re-run this notebook to fold them in.

Clip numbers are reassigned sequentially per `(subject, level)`, so duplicate-download files
like `… clip_03 (1).mp4` just become the next clip instead of colliding. A short `README.md`
is written into `raw_videos_binary/` describing the result (there is no separate manifest).


In [ ]:
# --- Setup: mount Drive, paths, the fixed label collapse ---
import os, re, glob, shutil
from collections import defaultdict
from datetime import datetime

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_FOLDER = "/content/drive/MyDrive/Argus"
except ImportError:
    # Not in Colab: point ARGUS_PROJECT_FOLDER at a path that has dataset/raw_videos/ under it.
    PROJECT_FOLDER = os.environ.get("ARGUS_PROJECT_FOLDER", "./Argus")

RAW_DIR    = f"{PROJECT_FOLDER}/dataset/raw_videos"
BINARY_DIR = f"{PROJECT_FOLDER}/dataset/raw_videos_binary"
VIDEO_EXT  = ".mp4"

# The collapse is fixed: Alert(1) + Low Vigilant(2) -> Not Drowsy(1); Drowsy(3) -> Drowsy(2).
# This is the "drowsy_vs_not" binary framing; class names match CLASS_NAMES in notebooks 01/02/06/09.
LEVEL_TO_BINARY    = {1: 1, 2: 1, 3: 2}
BINARY_CLASS_NAMES = {1: "Not Drowsy", 2: "Drowsy"}

assert os.path.isdir(RAW_DIR), f"raw_videos/ not found at {RAW_DIR}"
print("Source :", RAW_DIR)
print("Target :", BINARY_DIR)
print("Collapse:", LEVEL_TO_BINARY, "->", BINARY_CLASS_NAMES)


In [ ]:
# --- Build raw_videos_binary/ ---
DUP_RE = re.compile(r"\((\d+)\)\s*$")   # trailing " (1)" that Drive/downloads add to copies

def parse(path):
    """subject_07/level_2_clip_03 (1).mp4 -> dict(subject, level=2, clip=3, dup=1, ...)."""
    stem    = os.path.splitext(os.path.basename(path))[0]
    subject = os.path.basename(os.path.dirname(path))
    lm = re.search(r"level_(\d+)", stem, re.I)
    cm = re.search(r"clip_(\d+)",  stem, re.I)
    if not lm or not cm:
        return None
    dm = DUP_RE.search(stem)
    return dict(subject=subject, level=int(lm.group(1)), clip=int(cm.group(1)),
                dup=int(dm.group(1)) if dm else 0, stem=stem, path=path)

parsed, skipped = [], []
for p in glob.glob(f"{RAW_DIR}/**/*{VIDEO_EXT}", recursive=True):
    r = parse(p)
    if r is None:
        skipped.append(p)
    elif r["level"] not in LEVEL_TO_BINARY:
        raise ValueError(f"{p}: level {r['level']} outside 1-3 — unexpected.")
    else:
        parsed.append(r)

# Order so "clip_03 (1).mp4" lands right after "clip_03.mp4", then number sequentially.
parsed.sort(key=lambda r: (r["subject"], r["level"], r["clip"], r["dup"], r["stem"]))

counter, plan = defaultdict(int), []
for r in parsed:
    bl = LEVEL_TO_BINARY[r["level"]]
    counter[(r["subject"], bl)] += 1
    dst = os.path.join(BINARY_DIR, r["subject"],
                       f"level_{bl}_clip_{counter[(r['subject'], bl)]:02d}{VIDEO_EXT}")
    plan.append((r["path"], dst, bl, r["dup"] > 0))

keep = {dst for _, dst, _, _ in plan}
copied = unchanged = removed = 0
for src, dst, _, _ in plan:
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src):
        unchanged += 1
    else:
        shutil.copy2(src, dst)
        copied += 1
for p in glob.glob(f"{BINARY_DIR}/**/*{VIDEO_EXT}", recursive=True):   # drop stale files from old runs
    if p not in keep:
        os.remove(p)
        removed += 1

per_class = defaultdict(int)
for _, _, bl, _ in plan:
    per_class[bl] += 1
n_subjects = len({s for s, _ in counter})
n_dups     = sum(1 for _, _, _, is_dup in plan if is_dup)

print(f"{len(plan)} clips  ->  copied {copied}, unchanged {unchanged}, removed {removed} stale")
print(f"'(N)' duplicate-download files folded in as normal clips: {n_dups}")
if skipped:
    print(f"skipped {len(skipped)} file(s) with an unparseable name:")
    for p in skipped[:15]:
        print("  -", os.path.relpath(p, RAW_DIR))

readme = f"""# raw_videos_binary/

2-class copy of `../raw_videos/`, generated by `notebook/relabel_binary_raw_videos.ipynb`
on {datetime.now():%Y-%m-%d %H:%M}. **Do not edit by hand** — re-run that notebook. Safe to
delete this whole folder; nothing here is unique.

## Label collapse

| source level | this tree | class |
|---|---|---|
| 1  Alert         | level_1 | Not Drowsy |
| 2  Low Vigilant  | level_1 | Not Drowsy |
| 3  Drowsy        | level_2 | Drowsy |

`level_1` = "{BINARY_CLASS_NAMES[1]}", `level_2` = "{BINARY_CLASS_NAMES[2]}" -- the
`drowsy_vs_not` framing: `Alert` and `Low Vigilant` both become `Not Drowsy`.

## Notes

- `subject_NN/level_<1-2>_clip_<NN>.mp4`, same structure as `raw_videos/`.
- Clip numbers are reassigned sequentially per (subject, level); they do NOT match the source
  clip numbers 1:1.
- `(N)`-suffixed duplicate-download files in the source are folded in as extra clips.

## Contents ({datetime.now():%Y-%m-%d})

- Subjects: {n_subjects}
- {BINARY_CLASS_NAMES[1]} (level_1): {per_class[1]} clips
- {BINARY_CLASS_NAMES[2]} (level_2): {per_class[2]} clips
- Total: {per_class[1] + per_class[2]} clips
"""
os.makedirs(BINARY_DIR, exist_ok=True)
with open(os.path.join(BINARY_DIR, "README.md"), "w") as f:
    f.write(readme)
print("wrote", os.path.join(BINARY_DIR, "README.md"))


In [ ]:
# --- List raw_videos_binary/ ---
out = sorted(glob.glob(f"{BINARY_DIR}/**/*{VIDEO_EXT}", recursive=True))
by_subject = defaultdict(lambda: defaultdict(int))
bad = []
for p in out:
    subject = os.path.basename(os.path.dirname(p))
    lvl = int(re.search(r"level_(\d+)", os.path.basename(p)).group(1))
    if lvl not in (1, 2):
        bad.append(p)
    by_subject[subject][lvl] += 1

tot = defaultdict(int)
print(BINARY_DIR, "\n")
for subject in sorted(by_subject):
    c = by_subject[subject]
    tot[1] += c[1]; tot[2] += c[2]
    print(f"  {subject:<14} level_1={c[1]:>3}   level_2={c[2]:>3}")
print(f"\n  {'TOTAL':<14} level_1={tot[1]:>3}   level_2={tot[2]:>3}"
      f"    ({tot[1] + tot[2]} clips across {len(by_subject)} subjects)")
print(f"\n  level_1 = {BINARY_CLASS_NAMES[1]}")
print(f"  level_2 = {BINARY_CLASS_NAMES[2]}")
assert not bad, f"unexpected non-1/2 levels: {bad[:5]}"
